In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
ann = pd.read_csv("../data/processed/mhs_main_experiment_annotations_with_split.csv")

print("Annotation rows:", ann.shape)
print("Unique comments:", ann["comment_id"].nunique())

Annotation rows: (49433, 53)
Unique comments: 19761


In [3]:
LABELS = [0, 1, 2]
label_cols = [f"hatespeech_{label}_prob" for label in LABELS]

def make_distribution(group):
    counts = group["hatespeech"].value_counts(normalize=True)
    return np.array([counts.get(label, 0.0) for label in LABELS])

def entropy_np(probs):
    probs = np.array(probs)
    probs = probs[probs > 0]
    if len(probs) == 0:
        return 0.0
    return -np.sum(probs * np.log2(probs))

In [4]:
rows = []

for comment_id, group in ann.groupby("comment_id"):
    dist = make_distribution(group)
    ent = entropy_np(dist)

    rows.append({
        "comment_id": comment_id,
        "split": group["split"].iloc[0],
        "text_clean": group["text_clean"].iloc[0],
        "annotation_count": len(group),
        "hatespeech_0_prob": dist[0],
        "hatespeech_1_prob": dist[1],
        "hatespeech_2_prob": dist[2],
        "hatespeech_entropy": ent,
        "majority_label": int(np.argmax(dist)),
        "distribution_pattern": tuple(np.round(dist, 3)),
        "is_zero_entropy": ent == 0
    })

dist_audit_df = pd.DataFrame(rows)

print(dist_audit_df.shape)
dist_audit_df.head()

(19761, 11)


,comment_id,split,text_clean,annotation_count,hatespeech_0_prob,hatespeech_1_prob,hatespeech_2_prob,hatespeech_entropy,majority_label,distribution_pattern,is_zero_entropy
0,6,train,First off you look cool as fuck! Anyway if we ...,2,1.000000,0.0,0.000000,-0.000000,0,"(1.0, 0.0, 0.0)",True
1,7,test,\*points to posters asking for palestinian rig...,2,0.500000,0.0,0.500000,1.000000,0,"(0.5, 0.0, 0.5)",False
2,10,train,"They'll come back in your plan, also. Plus we ...",3,0.333333,0.0,0.666667,0.918296,2,"(0.333, 0.0, 0.667)",False
3,11,validation,"eat my fuck, bitch",2,0.000000,0.5,0.500000,1.000000,1,"(0.0, 0.5, 0.5)",False
4,12,train,She ugly anyways,2,0.500000,0.0,0.500000,1.000000,0,"(0.5, 0.0, 0.5)",False


In [5]:
pattern_summary = (
    dist_audit_df["distribution_pattern"]
    .value_counts()
    .reset_index()
)

pattern_summary.columns = ["distribution_pattern", "comments"]
pattern_summary["percent"] = (
    pattern_summary["comments"] / len(dist_audit_df) * 100
).round(2)

pattern_summary.head(30)

,distribution_pattern,comments,percent
0,"(1.0, 0.0, 0.0)",11082,56.08
1,"(0.0, 0.0, 1.0)",3366,17.03
2,"(0.5, 0.0, 0.5)",1325,6.71
3,"(0.0, 1.0, 0.0)",858,4.34
4,"(0.5, 0.5, 0.0)",598,3.03
5,"(0.667, 0.0, 0.333)",569,2.88
6,"(0.333, 0.0, 0.667)",412,2.08
7,"(0.0, 0.5, 0.5)",296,1.50
8,"(0.667, 0.333, 0.0)",287,1.45
9,"(0.333, 0.333, 0.333)",205,1.04


In [6]:
unique_pattern_summary = pd.DataFrame([
    {
        "subset": "all_comments",
        "comments": len(dist_audit_df),
        "unique_patterns": dist_audit_df["distribution_pattern"].nunique(),
        "top_pattern_percent": pattern_summary.iloc[0]["percent"],
        "top_5_patterns_percent": pattern_summary.head(5)["percent"].sum(),
        "top_10_patterns_percent": pattern_summary.head(10)["percent"].sum(),
    }
])

unique_pattern_summary

,subset,comments,unique_patterns,top_pattern_percent,top_5_patterns_percent,top_10_patterns_percent
0,all_comments,19761,83,56.08,87.19,96.14


In [7]:
threshold_rows = []

for threshold in [1, 2, 3, 4, 5]:
    subset = dist_audit_df[
        dist_audit_df["annotation_count"] >= threshold
    ].copy()

    pattern_counts = subset["distribution_pattern"].value_counts()

    threshold_rows.append({
        "annotation_threshold": f">={threshold}",
        "comments": len(subset),
        "percent_of_all": round(len(subset) / len(dist_audit_df) * 100, 2),
        "unique_patterns": subset["distribution_pattern"].nunique(),
        "zero_entropy_percent": round((subset["hatespeech_entropy"] == 0).mean() * 100, 2),
        "mean_entropy": subset["hatespeech_entropy"].mean(),
        "median_entropy": subset["hatespeech_entropy"].median(),
        "top_pattern": pattern_counts.index[0] if len(pattern_counts) > 0 else None,
        "top_pattern_percent": round(pattern_counts.iloc[0] / len(subset) * 100, 2) if len(subset) > 0 else None,
        "top_5_patterns_percent": round(pattern_counts.head(5).sum() / len(subset) * 100, 2) if len(subset) > 0 else None,
        "top_10_patterns_percent": round(pattern_counts.head(10).sum() / len(subset) * 100, 2) if len(subset) > 0 else None,
    })

threshold_pattern_summary = pd.DataFrame(threshold_rows)
threshold_pattern_summary

,annotation_threshold,comments,percent_of_all,unique_patterns,zero_entropy_percent,mean_entropy,median_entropy,top_pattern,top_pattern_percent,top_5_patterns_percent,top_10_patterns_percent
0,>=1,19761,100.00,83,77.46,0.225016,0.000000,"(1.0, 0.0, 0.0)",56.08,87.19,96.14
1,>=2,10639,53.84,83,58.13,0.417948,0.000000,"(1.0, 0.0, 0.0)",47.98,80.89,93.57
2,>=3,4644,23.50,83,49.35,0.504641,0.721928,"(1.0, 0.0, 0.0)",43.07,76.59,90.65
3,>=4,1235,6.25,75,43.24,0.559334,0.811278,"(1.0, 0.0, 0.0)",38.38,72.87,93.04
4,>=5,87,0.44,63,20.69,0.615387,0.677404,"(1.0, 0.0, 0.0)",17.24,29.89,39.08


In [8]:
split_threshold_rows = []

for split in ["train", "validation", "test"]:
    split_df = dist_audit_df[dist_audit_df["split"] == split]

    for threshold in [1, 2, 3]:
        subset = split_df[split_df["annotation_count"] >= threshold]

        pattern_counts = subset["distribution_pattern"].value_counts()

        split_threshold_rows.append({
            "split": split,
            "annotation_threshold": f">={threshold}",
            "comments": len(subset),
            "unique_patterns": subset["distribution_pattern"].nunique(),
            "zero_entropy_percent": round((subset["hatespeech_entropy"] == 0).mean() * 100, 2),
            "mean_entropy": subset["hatespeech_entropy"].mean(),
            "top_pattern_percent": round(pattern_counts.iloc[0] / len(subset) * 100, 2) if len(subset) > 0 else None,
            "top_5_patterns_percent": round(pattern_counts.head(5).sum() / len(subset) * 100, 2) if len(subset) > 0 else None
        })

split_threshold_pattern_summary = pd.DataFrame(split_threshold_rows)
split_threshold_pattern_summary

,split,annotation_threshold,comments,unique_patterns,zero_entropy_percent,mean_entropy,top_pattern_percent,top_5_patterns_percent
0,train,>=1,13832,68,77.56,0.223532,56.28,87.23
1,train,>=2,7437,68,58.26,0.415744,48.20,81.05
2,train,>=3,3252,68,49.63,0.499966,43.33,76.97
3,validation,>=1,2964,29,77.13,0.228493,56.17,87.11
4,validation,>=2,1601,29,57.65,0.423018,47.53,80.76
5,validation,>=3,687,29,49.49,0.504006,43.96,75.98
6,test,>=1,2965,29,77.30,0.228469,55.04,87.05
7,test,>=2,1601,29,57.96,0.423117,47.41,80.26
8,test,>=3,705,27,47.94,0.526824,40.99,75.46


In [9]:
nonzero_df = dist_audit_df[dist_audit_df["hatespeech_entropy"] > 0].copy()

nonzero_pattern_counts = (
    nonzero_df["distribution_pattern"]
    .value_counts()
    .reset_index()
)

nonzero_pattern_counts.columns = ["distribution_pattern", "comments"]
nonzero_pattern_counts["percent"] = (
    nonzero_pattern_counts["comments"] / len(nonzero_df) * 100
).round(2)

print("Non-zero entropy comments:", len(nonzero_df))
print("Percent of all:", round(len(nonzero_df) / len(dist_audit_df) * 100, 2))
print("Unique patterns:", nonzero_df["distribution_pattern"].nunique())

nonzero_pattern_counts.head(30)

Non-zero entropy comments: 4455
Percent of all: 22.54
Unique patterns: 80


,distribution_pattern,comments,percent
0,"(0.5, 0.0, 0.5)",1325,29.74
1,"(0.5, 0.5, 0.0)",598,13.42
2,"(0.667, 0.0, 0.333)",569,12.77
3,"(0.333, 0.0, 0.667)",412,9.25
4,"(0.0, 0.5, 0.5)",296,6.64
5,"(0.667, 0.333, 0.0)",287,6.44
6,"(0.333, 0.333, 0.333)",205,4.60
7,"(0.75, 0.0, 0.25)",149,3.34
8,"(0.0, 0.333, 0.667)",110,2.47
9,"(0.75, 0.25, 0.0)",94,2.11


In [10]:
nonzero_threshold_rows = []

for threshold in [2, 3, 4, 5]:
    subset = dist_audit_df[
        (dist_audit_df["annotation_count"] >= threshold) &
        (dist_audit_df["hatespeech_entropy"] > 0)
    ].copy()

    pattern_counts = subset["distribution_pattern"].value_counts()

    nonzero_threshold_rows.append({
        "subset": f"nonzero_entropy_and_annotations>={threshold}",
        "comments": len(subset),
        "percent_of_all": round(len(subset) / len(dist_audit_df) * 100, 2),
        "unique_patterns": subset["distribution_pattern"].nunique(),
        "mean_entropy": subset["hatespeech_entropy"].mean(),
        "median_entropy": subset["hatespeech_entropy"].median(),
        "top_pattern_percent": round(pattern_counts.iloc[0] / len(subset) * 100, 2) if len(subset) > 0 else None,
        "top_5_patterns_percent": round(pattern_counts.head(5).sum() / len(subset) * 100, 2) if len(subset) > 0 else None,
        "top_10_patterns_percent": round(pattern_counts.head(10).sum() / len(subset) * 100, 2) if len(subset) > 0 else None,
    })

nonzero_threshold_summary = pd.DataFrame(nonzero_threshold_rows)
nonzero_threshold_summary

,subset,comments,percent_of_all,unique_patterns,mean_entropy,median_entropy,top_pattern_percent,top_5_patterns_percent,top_10_patterns_percent
0,nonzero_entropy_and_annotations>=2,4455,22.54,80,0.998103,1.000000,29.74,71.83,90.80
1,nonzero_entropy_and_annotations>=3,2352,11.90,80,0.996408,0.918296,24.19,68.96,88.69
2,nonzero_entropy_and_annotations>=4,701,3.55,73,0.985418,0.811278,21.26,71.75,89.73
3,nonzero_entropy_and_annotations>=5,69,0.35,61,0.775923,0.774243,4.35,17.39,26.09


In [11]:
subset_defs = {
    "all_comments": dist_audit_df,
    "annotations>=2": dist_audit_df[dist_audit_df["annotation_count"] >= 2],
    "annotations>=3": dist_audit_df[dist_audit_df["annotation_count"] >= 3],
    "nonzero_entropy": dist_audit_df[dist_audit_df["hatespeech_entropy"] > 0],
    "nonzero_entropy_and_annotations>=2": dist_audit_df[
        (dist_audit_df["hatespeech_entropy"] > 0) &
        (dist_audit_df["annotation_count"] >= 2)
    ],
    "nonzero_entropy_and_annotations>=3": dist_audit_df[
        (dist_audit_df["hatespeech_entropy"] > 0) &
        (dist_audit_df["annotation_count"] >= 3)
    ],
}

majority_rows = []

for subset_name, subset in subset_defs.items():
    counts = subset["majority_label"].value_counts(normalize=True)

    majority_rows.append({
        "subset": subset_name,
        "comments": len(subset),
        "majority_0_percent": round(counts.get(0, 0) * 100, 2),
        "majority_1_percent": round(counts.get(1, 0) * 100, 2),
        "majority_2_percent": round(counts.get(2, 0) * 100, 2),
    })

majority_subset_summary = pd.DataFrame(majority_rows)
majority_subset_summary

,subset,comments,majority_0_percent,majority_1_percent,majority_2_percent
0,all_comments,19761,72.95,6.26,20.79
1,annotations>=2,10639,79.32,4.22,16.46
2,annotations>=3,4644,75.90,1.89,22.20
3,nonzero_entropy,4455,74.84,8.51,16.66
4,nonzero_entropy_and_annotations>=2,4455,74.84,8.51,16.66
5,nonzero_entropy_and_annotations>=3,2352,64.84,3.61,31.55


In [12]:
dist_audit_df["pattern_type"] = np.where(
    dist_audit_df["hatespeech_entropy"] == 0,
    "one_hot_consensus",
    "mixed_distribution"
)

pattern_type_summary = (
    dist_audit_df
    .groupby("pattern_type")
    .agg(
        comments=("comment_id", "count"),
        mean_annotation_count=("annotation_count", "mean"),
        median_annotation_count=("annotation_count", "median"),
        mean_entropy=("hatespeech_entropy", "mean"),
        majority_0_percent=("majority_label", lambda x: round((x == 0).mean() * 100, 2)),
        majority_1_percent=("majority_label", lambda x: round((x == 1).mean() * 100, 2)),
        majority_2_percent=("majority_label", lambda x: round((x == 2).mean() * 100, 2)),
    )
    .reset_index()
)

pattern_type_summary

,pattern_type,comments,mean_annotation_count,median_annotation_count,mean_entropy,majority_0_percent,majority_1_percent,majority_2_percent
0,mixed_distribution,4455,5.593490,3.0,0.998103,74.84,8.51,16.66
1,one_hot_consensus,15306,1.601594,1.0,0.000000,72.40,5.61,21.99


In [13]:
common_mixed_patterns = (
    dist_audit_df[dist_audit_df["hatespeech_entropy"] > 0]
    ["distribution_pattern"]
    .value_counts()
    .head(10)
    .index
)

example_rows = []

for pattern in common_mixed_patterns:
    examples = dist_audit_df[
        dist_audit_df["distribution_pattern"] == pattern
    ].head(3)

    for _, row in examples.iterrows():
        example_rows.append({
            "distribution_pattern": pattern,
            "comment_id": row["comment_id"],
            "split": row["split"],
            "annotation_count": row["annotation_count"],
            "entropy": row["hatespeech_entropy"],
            "majority_label": row["majority_label"],
            "text_clean": row["text_clean"][:300]
        })

mixed_pattern_examples = pd.DataFrame(example_rows)
mixed_pattern_examples

,distribution_pattern,comment_id,split,annotation_count,entropy,majority_label,text_clean
0,"(0.5, 0.0, 0.5)",7,test,2,1.000000,0,\*points to posters asking for palestinian rig...
1,"(0.5, 0.0, 0.5)",12,train,2,1.000000,0,She ugly anyways
2,"(0.5, 0.0, 0.5)",22,train,2,1.000000,0,The guys are one thing but then you have a wom...
3,"(0.5, 0.5, 0.0)",88,train,2,1.000000,0,Better get 2 abortions for your spawn cause yo...
4,"(0.5, 0.5, 0.0)",186,train,2,1.000000,0,Um if a penis is triggering the deeper fears o...
5,"(0.5, 0.5, 0.0)",202,validation,2,1.000000,0,More conflation of sex and gender.. I. Am. Sho...
6,"(0.667, 0.0, 0.333)",26,train,3,0.918296,0,I'd love to rip those panties off and shove my...
7,"(0.667, 0.0, 0.333)",132,validation,3,0.918296,0,"As someone from the US, fuck our craven sack o..."
8,"(0.667, 0.0, 0.333)",224,train,3,0.918296,0,I fought so my daughters could be sluts!
9,"(0.333, 0.0, 0.667)",10,train,3,0.918296,2,"They'll come back in your plan, also. Plus we ..."


In [15]:
os.makedirs("../results/tables", exist_ok=True)

report_path = "../results/tables/hatespeech_distribution_pattern_audit.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("HATESPEECH DISTRIBUTION PATTERN AUDIT\n")
    f.write("=" * 90 + "\n\n")

    f.write("1. DATASET SIZE\n")
    f.write("-" * 90 + "\n")
    f.write(f"Unique comments: {len(dist_audit_df)}\n")
    f.write(f"Annotation rows: {len(ann)}\n\n")

    f.write("2. OVERALL MOST COMMON DISTRIBUTION PATTERNS\n")
    f.write("-" * 90 + "\n")
    f.write(pattern_summary.head(30).to_string(index=False))
    f.write("\n\n")

    f.write("3. UNIQUE PATTERN SUMMARY\n")
    f.write("-" * 90 + "\n")
    f.write(unique_pattern_summary.to_string(index=False))
    f.write("\n\n")

    f.write("4. PATTERN SUMMARY BY ANNOTATION THRESHOLD\n")
    f.write("-" * 90 + "\n")
    f.write(threshold_pattern_summary.to_string(index=False))
    f.write("\n\n")

    f.write("5. PATTERN SUMMARY BY SPLIT AND THRESHOLD\n")
    f.write("-" * 90 + "\n")
    f.write(split_threshold_pattern_summary.to_string(index=False))
    f.write("\n\n")

    f.write("6. NON-ZERO ENTROPY PATTERNS\n")
    f.write("-" * 90 + "\n")
    f.write(nonzero_pattern_counts.head(30).to_string(index=False))
    f.write("\n\n")

    f.write("7. NON-ZERO ENTROPY + ANNOTATION THRESHOLD SUMMARY\n")
    f.write("-" * 90 + "\n")
    f.write(nonzero_threshold_summary.to_string(index=False))
    f.write("\n\n")

    f.write("8. MAJORITY LABEL DISTRIBUTION BY SUBSET\n")
    f.write("-" * 90 + "\n")
    f.write(majority_subset_summary.to_string(index=False))
    f.write("\n\n")

    f.write("9. PATTERN TYPE SUMMARY\n")
    f.write("-" * 90 + "\n")
    f.write(pattern_type_summary.to_string(index=False))
    f.write("\n\n")

    f.write("10. COMMON MIXED PATTERN EXAMPLES\n")
    f.write("-" * 90 + "\n")
    f.write(mixed_pattern_examples.to_string(index=False))
    f.write("\n\n")


print("Saved:", report_path)
print(open(report_path, encoding="utf-8").read())

Saved: ../results/tables/hatespeech_distribution_pattern_audit.txt
HATESPEECH DISTRIBUTION PATTERN AUDIT

1. DATASET SIZE
------------------------------------------------------------------------------------------
Unique comments: 19761
Annotation rows: 49433

2. OVERALL MOST COMMON DISTRIBUTION PATTERNS
------------------------------------------------------------------------------------------
 distribution_pattern  comments  percent
      (1.0, 0.0, 0.0)     11082    56.08
      (0.0, 0.0, 1.0)      3366    17.03
      (0.5, 0.0, 0.5)      1325     6.71
      (0.0, 1.0, 0.0)       858     4.34
      (0.5, 0.5, 0.0)       598     3.03
  (0.667, 0.0, 0.333)       569     2.88
  (0.333, 0.0, 0.667)       412     2.08
      (0.0, 0.5, 0.5)       296     1.50
  (0.667, 0.333, 0.0)       287     1.45
(0.333, 0.333, 0.333)       205     1.04
    (0.75, 0.0, 0.25)       149     0.75
  (0.0, 0.333, 0.667)       110     0.56
    (0.75, 0.25, 0.0)        94     0.48
    (0.25, 0.0, 0.75)        8